In [1]:
import pandas as pd
from konlpy.tag import Mecab
from gensim import corpora
from gensim.models.ldamodel import LdaModel
import networkx as nx
import numpy as np
import tqdm

In [2]:
# --- 1단계: 데이터 준비 및 전처리 ---
print("1. 데이터 준비 및 전처리 시작...")


1. 데이터 준비 및 전처리 시작...


In [3]:
KINDS_PATH = '../data/interim/news/deep_search_news.csv'
STOPWORD_PATH = '../data/raw/news/stopwords-ko.txt'

POSITIVE_PATH = '../data/raw/news/sentiment/positive.txt'
NEGATIVE_PATH = '../data/raw/news/sentiment/negative.txt'
NATURAL_PATH = '../data/raw/news/sentiment/natural.txt'
# 뉴스 데이터 읽기
df = pd.read_csv(KINDS_PATH)
#df = data.head(20) # 테스트용으로 상위 20개만 해봄

# 각종 TXT 파일 불러오기
def load_txt(PATH):
    with open(PATH, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]
# 불용어 불러오기
stopwords = load_txt(STOPWORD_PATH)
# 긍정, 부정, 중립 단어 불러오기
positive = load_txt(POSITIVE_PATH)
negative = load_txt(NEGATIVE_PATH)
natural = load_txt(NATURAL_PATH)

print(f"불용어 단어 예시 : {stopwords[:5]}...")
print(f"긍정 단어 예시 : {positive[:5]}...")
print(f"부정 단어 예시 : {negative[:5]}...")
print(f"중립 단어 예시 : {natural[:5]}...")

불용어 단어 예시 : ['가', '가까스로', '가령', '각', '각각']...
긍정 단어 예시 : ['활황', '급매물', '소진', '강세', '매수세']...
부정 단어 예시 : ['침체', '급매', '투매', '하락', '폭락']...
중립 단어 예시 : ['부동산', '아파트', '주택', '토지', '건물']...


In [7]:
print(f"수집된 기사의 갯수는 {len(df)}개 입니다.")

수집된 기사의 갯수는 37437개 입니다.


In [8]:
df['content']

0        유의주 기자 = 8일 오후 6시 52분께 대전시 지족동의 한 아파트단지 지하 2층 ...
1        특히 지난해부터 본격 하락세에 접어든 울산 동구의 집값이 지난달 현대중공업이 해양사...
2        거제 고현항 항만재개발 사업 조감도.\n\n거제 고현항 재개발 사업은 낙후된 항구 ...
3        ■토지(매매)=부산 강서구 범방동 1029㎡(311평), 준주거지역, 대로변, 매매...
4        이번 조치로 다주택자와 1주택자는 보유세 부담 때문에 추가적인 주택 투자를 망설이게...
                               ...                        
37432    등기부 등본상 근저당 6건, 가압류 3건, 압류 1건 등은 매각 후 잔금 납부 시 ...
37433    "농산물 도매거래 절반, 2030년까지 온라인으로 전환 목표" 정부는 농산물 유통의...
37434    또 중대재해가 발생한 건설사 등에 경제적 제재가 가해지면 소비자에게 피해가 갈 수 ...
37435    현대건설이 광명뉴타운 내 최대 규모, 최고 입지로 평가받는 광명11R주택재개발사업(...
37436    Sh수협은행은 지난 12일 이사회를 열고 트리니티자산운용 인수 추진안건을 의결했다고...
Name: content, Length: 37437, dtype: object

In [9]:
# 명사 추출 + 불용어 제거 함수
mecab = Mecab()
def tokenize(text):
    if not isinstance(text, str):
        return []
    return [word for word in mecab.nouns(text) 
            if len(word) > 1 and word not in stopwords]

# 토큰화 + 불용어 제거 적용
df['tokens'] = df['content'].apply(tokenize)

print(f"토큰 예시 : {df['tokens'][0]}")

토큰 예시 : ['오후', '대전시', '지족동', '아파트', '지하', '주차장', '주차', '승용차', '차량', '아파트', '경비원', '진화']


In [10]:
## --- 2단계: 토픽 모델링 및 텍스트랭크 ---
print("\n2. 토픽 모델링 및 텍스트랭크 시작...")


2. 토픽 모델링 및 텍스트랭크 시작...


In [11]:
# 토픽 모델링 (LDA)
dictionary = corpora.Dictionary(df['tokens'])
corpus = [dictionary.doc2bow(tokens) for tokens in df['tokens']]
lda_model = LdaModel(corpus, num_topics=8, id2word=dictionary, passes=15)

topics = lda_model.print_topics(num_words=30)
print(f"토픽 모델링 결과 예시 :\n {topics[0]}")

토픽 모델링 결과 예시 :
 (0, '0.032*"투자" + 0.022*"금융" + 0.016*"거래" + 0.016*"주식" + 0.015*"국내" + 0.015*"시장" + 0.012*"자산" + 0.011*"증권" + 0.010*"투자자" + 0.009*"서비스" + 0.009*"기업" + 0.009*"매매" + 0.009*"수익" + 0.008*"상품" + 0.008*"종목" + 0.008*"미국" + 0.007*"외국인" + 0.007*"거래소" + 0.007*"해외" + 0.006*"주가" + 0.006*"매수" + 0.006*"증권사" + 0.006*"상장" + 0.005*"개인" + 0.005*"은행" + 0.005*"가능" + 0.005*"규모" + 0.005*"지난해" + 0.005*"채권" + 0.005*"펀드"')


In [12]:
# 텍스트랭크
def text_rank_keywords(tokens):
    g = nx.Graph()
    for i in range(len(tokens) - 1):
        g.add_edge(tokens[i], tokens[i+1])
    pr = nx.pagerank(g, weight='weight')
    return sorted(pr, key=pr.get, reverse=True) # 상위 몇개를 포함할건가?는 논문에 없다

In [13]:
df['textrank_keywords'] = df['tokens'].apply(text_rank_keywords)
print("\n텍스트랭크 키워드:")
print(df[['content', 'textrank_keywords']].head())


텍스트랭크 키워드:
                                             content  \
0  유의주 기자 = 8일 오후 6시 52분께 대전시 지족동의 한 아파트단지 지하 2층 ...   
1  특히 지난해부터 본격 하락세에 접어든 울산 동구의 집값이 지난달 현대중공업이 해양사...   
2  거제 고현항 항만재개발 사업 조감도.\n\n거제 고현항 재개발 사업은 낙후된 항구 ...   
3  ■토지(매매)=부산 강서구 범방동 1029㎡(311평), 준주거지역, 대로변, 매매...   
4  이번 조치로 다주택자와 1주택자는 보유세 부담 때문에 추가적인 주택 투자를 망설이게...   

                                   textrank_keywords  
0  [아파트, 대전시, 경비원, 지족동, 주차, 주차장, 승용차, 지하, 차량, 오후,...  
1  [중공업, 지난달, 동구, 현대, 평균, 울산시, 기록, 본격, 중단, 발표, 가동...  
2  [사업, 조성, 부지, 거제, 고현항, 개발, 녹지, 주거, 공원, 문화, 부족, ...  
3  [매매, 산리, 전원, 삼익, 비치, 생면, 주택, 남천동, 욕실, 주거, 지역, ...  
4  [주택, 시장, 부담, 종부, 인상, 반사, 건물, 상업, 소형, 관망세, 작용, ...  


In [14]:
# --- 3단계: 감성 사전 기반 감성 점수 산출 ---
print("\n3. 감성 사전 기반 감성 점수 산출 시작...")

def get_sentiment_score(tokens):
    pos_score = sum(1 for word in tokens if word in positive)
    neg_score = sum(1 for word in tokens if word in negative)
    nat_score = sum(1 for word in tokens if word in natural)
    total_words = len(tokens)
    if total_words == 0:
        return 0
    return (pos_score - neg_score) / total_words

df['sentiment_score'] = df['tokens'].apply(get_sentiment_score)

print("\n감성 사전 기반 감성 점수:")
print(df[['content', 'sentiment_score']].head())


3. 감성 사전 기반 감성 점수 산출 시작...

감성 사전 기반 감성 점수:
                                             content  sentiment_score
0  유의주 기자 = 8일 오후 6시 52분께 대전시 지족동의 한 아파트단지 지하 2층 ...         0.000000
1  특히 지난해부터 본격 하락세에 접어든 울산 동구의 집값이 지난달 현대중공업이 해양사...        -0.075000
2  거제 고현항 항만재개발 사업 조감도.\n\n거제 고현항 재개발 사업은 낙후된 항구 ...         0.090909
3  ■토지(매매)=부산 강서구 범방동 1029㎡(311평), 준주거지역, 대로변, 매매...         0.240000
4  이번 조치로 다주택자와 1주택자는 보유세 부담 때문에 추가적인 주택 투자를 망설이게...        -0.187500


In [15]:
# --- 4단계: 월별 감성 지수 산출 및 예측 모델 통합 ---
print("\n4. 월별 감성 지수 산출 및 예측 모델 통합...")


4. 월별 감성 지수 산출 및 예측 모델 통합...


In [16]:
# datetime 변환
df['date'] = pd.to_datetime(df['date'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
df['month'] = df['date'].dt.to_period('M')

In [17]:
monthly_sentiment = df.groupby('month')['sentiment_score'].mean().reset_index()
monthly_sentiment['month'] = monthly_sentiment['month'].astype(str)

# monthly_sentiment month 컬럼도 period[M]로 변환
monthly_sentiment['month'] = pd.to_datetime(monthly_sentiment['month']).dt.to_period('M')


In [18]:
print("\n월별 감성 지수:")
print(monthly_sentiment)
monthly_sentiment


월별 감성 지수:
      month  sentiment_score
0   2018-07         0.016983
1   2018-08         0.035666
2   2018-09         0.013844
3   2018-10         0.023940
4   2018-11         0.004340
..      ...              ...
82  2025-05         0.045637
83  2025-06         0.003236
84  2025-07        -0.007471
85  2025-08         0.010208
86  2025-09         0.007458

[87 rows x 2 columns]


,month,sentiment_score
0,2018-07,0.016983
1,2018-08,0.035666
2,2018-09,0.013844
3,2018-10,0.023940
4,2018-11,0.004340
...,...,...
82,2025-05,0.045637
83,2025-06,0.003236
84,2025-07,-0.007471
85,2025-08,0.010208


In [19]:
SALE_PATH = '../data/interim/apt/gang_nam_apt_with_long_lat.csv'
sale = pd.read_csv(SALE_PATH)

In [20]:
sale.head(1)

,단지명,전용면적(㎡),층,건축년도,도로명,면적당 단가(만원),아파트 나이,구,계약일자,계약년월,alpha,경도,위도
0,대치하나빌,169.19,4,2003,삼성로57길 37-5,6.753467,15,강남구,2018-07-01,201807,0.5,127.058493,37.497035


In [21]:
# datetime 변환
sale['계약일자'] = pd.to_datetime(sale['계약일자'], errors='coerce')  # 변환 불가 값은 NaT 처리
# 월 단위 추출
sale['month'] = sale['계약일자'].dt.to_period('M')
drop_col = ['단지명','도로명','경도','위도']

In [22]:
sale.columns

Index(['단지명', '전용면적(㎡)', '층', '건축년도', '도로명', '면적당 단가(만원)', '아파트 나이', '구',
       '계약일자', '계약년월', 'alpha', '경도', '위도', 'month'],
      dtype='object')

In [23]:
sale.drop(drop_col, axis=1, inplace=True)

In [24]:
sale.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,구,계약일자,계약년월,alpha,month
0,169.1900,4,2003,6.753467,15,강남구,2018-07-01,201807,0.500000,2018-07
1,84.7300,5,1994,7.399156,24,강남구,2018-07-01,201807,0.464386,2018-07
2,147.9554,7,2006,7.337995,12,강남구,2018-07-02,201807,0.600000,2018-07
3,84.4300,10,1980,7.618163,38,강남구,2018-07-02,201807,0.000000,2018-07
4,76.7900,4,1979,7.570627,39,강남구,2018-07-02,201807,0.000000,2018-07


In [25]:
merged_df = pd.merge(sale, monthly_sentiment, on='month', how='left')

In [26]:
merged_df.drop('month', axis=1, inplace=True)

In [27]:
merged_df.head()

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,구,계약일자,계약년월,alpha,sentiment_score
0,169.1900,4,2003,6.753467,15,강남구,2018-07-01,201807,0.500000,0.016983
1,84.7300,5,1994,7.399156,24,강남구,2018-07-01,201807,0.464386,0.016983
2,147.9554,7,2006,7.337995,12,강남구,2018-07-02,201807,0.600000,0.016983
3,84.4300,10,1980,7.618163,38,강남구,2018-07-02,201807,0.000000,0.016983
4,76.7900,4,1979,7.570627,39,강남구,2018-07-02,201807,0.000000,0.016983


In [32]:
len(merged_df)

14575

In [33]:
merged_df.to_csv('../data/interim/gang_nam_sendimental_score_with_sale.csv',index=False)

In [30]:
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
import numpy as np


In [25]:

# 특성과 타깃 분리
X = merged_df[['전용면적(㎡)','층','건축년도','아파트 나이','alpha','sentiment_score']]
y = merged_df['면적당 단가(만원)']

# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('mlp', MLPRegressor(hidden_layer_sizes=(32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_absolute_error')

# MAE는 음수로 반환되므로 양수로 변환
mae_scores = -scores

print("10-fold CV MAE scores:", mae_scores)
print("Mean MAE:", np.mean(mae_scores))

10-fold CV MAE scores: [0.32901233 0.33292732 0.33243191 0.32902923 0.32926716 0.34074096
 0.33889897 0.33455075 0.33512781 0.33627461]
Mean MAE: 0.3338261047789882


In [26]:
sale

,전용면적(㎡),층,건축년도,면적당 단가(만원),아파트 나이,alpha,month
0,59.91,10,1998,6.766156,22,0.266667,2020-07
1,59.77,7,1996,7.499383,24,1.000000,2020-07
2,84.83,6,2013,7.401580,7,1.000000,2020-07
3,59.75,13,2016,7.066081,4,1.000000,2020-07
4,49.94,7,1989,6.967225,31,0.000000,2020-07
...,...,...,...,...,...,...,...
23568,54.34,2,1995,5.908227,30,0.000000,2025-06
23569,97.21,8,2006,7.406056,19,0.366667,2025-06
23570,23.70,11,2019,7.034407,6,0.800000,2025-06
23571,22.20,10,2002,7.139868,23,0.233333,2025-06


In [27]:

# 특성과 타깃 분리
X = sale[['전용면적(㎡)','층','건축년도','아파트 나이','alpha']]
y = sale['면적당 단가(만원)']

# MLP 회귀 모델 + 표준화
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('mlp', MLPRegressor(hidden_layer_sizes=(32,16,8),
                         activation='relu',
                         solver='adam',
                         max_iter=500,
                         random_state=42))
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 10-fold 교차검증 (MAE)
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='neg_mean_absolute_error')

# MAE는 음수로 반환되므로 양수로 변환
mae_scores = -scores

print("10-fold CV MAE scores:", mae_scores)
print("Mean MAE:", np.mean(mae_scores))

10-fold CV MAE scores: [0.3337105  0.33283429 0.3316904  0.33631928 0.33709114 0.33878993
 0.34406594 0.33747762 0.34000014 0.33981638]
Mean MAE: 0.3371795616117778


In [31]:
merged_df['sentiment_score'].to_csv('../data/interim/sendimental_score.csv', index=False)

In [31]:
len(merged_df)

14575

In [ ]:
# #경기종합지수
# EI = pd.read_csv(ECONOMIC_INDEX, encoding='cp949')
# #공종별 건설기성액
# CW = pd.read_csv(CONSTRUCTION_WORK, encoding='cp949')
# #부동산시장 소비심리지수
# RECI = pd.read_csv(REAL_ESTATE_CONSUMER_INDEX, encoding='cp949')
# #소비자 물가지수
# CPI = pd.read_csv(CPI_PATH, encoding='cp949')
# #주택시장 소비심리지수
# HCS = pd.read_csv(HOUSING_CONSUMER_SENTIMENT, encoding='cp949')
# #지역별 지가변동률
# PCR = pd.read_csv(PRICE_CHANGE_RATE, encoding='cp949')
# #토지시장 소비심리지수
# LMCSI = pd.read_csv(LAND_MARKET_CONSUMER_SENTIMENT_INDEX, encoding='cp949')
# #행정구역별 아파트 매매거래현황
# AT = pd.read_csv(APT_TRANSACTIONS, encoding='cp949')
# # 금리
# IR = pd.read_csv(INTEREST_RATE)
# IR = IR.iloc[:,13:]